# 🧠 CalRetail — Demand Forecasting
## Goal
Forecast daily product demand for any of the 5,000 SKUs using a single global XGBoost model
trained on the full, pre-engineered daily-sales feature table — with real calendar effects
(holidays, sale season, weekends) and honest, per-product accuracy metrics.

## Algorithmic Explanation
**Global multi-product XGBoost regression on lag + calendar + entity-baseline features**
1. Load `feature_daily_sales.csv` (built by `notebooks/feature_engineering.py`): real lag_7,
   lag_14, rolling_7/30-day means, and calendar flags for every product-day.
2. Add each product's and category's own average daily demand as features — this lets **one**
   global model generalise correctly across all products (each SKU's own scale is a feature),
   instead of fitting one small model per SKU and mis-applying it to other products.
3. Time-split train/test (train on the past, validate on the most recent 15%) and fit
   `XGBRegressor`. Report a real, per-product MAPE computed from that product's own held-out
   rows when available, falling back to the overall model MAPE — never a fabricated number.
4. Forecast recursively day-by-day, looking up real holiday/sale-season flags from
   `holiday_calendar.csv` for each future date.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

DOW_MAP = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}

# Use the pre-engineered daily sales feature table (real lags, rolling means and
# calendar flags computed in notebooks/feature_engineering.py) instead of
# recomputing a thin, single-product subset of features from raw transactions.
daily = load_table('feature_daily_sales')
daily['transaction_date'] = pd.to_datetime(daily['transaction_date'])
prod_ref = load_table('products')
daily = daily.merge(prod_ref[['product_id', 'category']], on='product_id', how='left')

holiday_cal = load_table('holiday_calendar')
holiday_cal['date'] = pd.to_datetime(holiday_cal['date'])
holiday_lookup = holiday_cal.set_index('date')[['is_holiday', 'is_sale_season']].to_dict('index')

daily['dow_num'] = daily['day_of_week'].map(DOW_MAP).fillna(0).astype(int)
for c in ['is_holiday', 'is_weekend', 'is_sale_season']:
    daily[c] = daily[c].astype(bool).astype(int)

# Per-product / per-category baseline demand levels. This lets ONE global
# model generalise across all 5,000 SKUs instead of fitting (and mis-applying)
# a separate small model per product.
product_avg = daily.groupby('product_id')['daily_qty'].mean().rename('product_avg_qty')
category_avg = daily.groupby('category')['daily_qty'].mean().rename('category_avg_qty')
daily = daily.merge(product_avg, on='product_id', how='left')
daily = daily.merge(category_avg, on='category', how='left')

FEATURES = ['lag_7', 'lag_14', 'rolling_7_mean', 'rolling_30_mean', 'dow_num', 'month',
            'day_of_month', 'week_number', 'quarter', 'is_holiday', 'is_weekend',
            'is_sale_season', 'product_avg_qty', 'category_avg_qty']

model_df = daily.dropna(subset=FEATURES + ['daily_qty']).sort_values('transaction_date')

# Time-based split (train on the past, test on the most recent 15%) — the only
# honest way to validate a forecasting model.
split_idx = int(len(model_df) * 0.85)
X_train, X_test = model_df[FEATURES].iloc[:split_idx], model_df[FEATURES].iloc[split_idx:]
y_train, y_test = model_df['daily_qty'].iloc[:split_idx], model_df['daily_qty'].iloc[split_idx:]

model = xgb.XGBRegressor(
    n_estimators=250, max_depth=5, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train)

test_preds = np.clip(model.predict(X_test), 0, None)
GLOBAL_MAE = float(mean_absolute_error(y_test, test_preds))
GLOBAL_MAPE = float(np.mean(np.abs((y_test - test_preds) / np.clip(y_test, 1, None))) * 100)

print(f"Global XGBoost demand model trained on {len(X_train):,} rows across {daily['product_id'].nunique():,} products.")
print(f"Held-out test MAE: {GLOBAL_MAE:.2f} units | MAPE: {GLOBAL_MAPE:.2f}%")

In [ ]:
_product_mape_cache = {}
_test_pid_lookup = model_df.loc[X_test.index, 'product_id']

def _product_test_mape(product_id):
    """Real MAPE computed from this product's own rows in the held-out test
    split when there are enough of them; otherwise the overall model MAPE.
    Never a fabricated/hashed number."""
    if product_id in _product_mape_cache:
        return _product_mape_cache[product_id]
    mask = _test_pid_lookup == product_id
    if mask.sum() >= 5:
        actual = y_test[mask]
        preds = np.clip(model.predict(X_test[mask]), 0, None)
        mape = float(np.mean(np.abs((actual - preds) / np.clip(actual, 1, None))) * 100)
    else:
        mape = GLOBAL_MAPE
    _product_mape_cache[product_id] = mape
    return mape


def get_demand_forecast(product_id, forecast_days=7):
    p_hist = daily[daily['product_id'] == product_id].sort_values('transaction_date')
    cat_series = prod_ref.loc[prod_ref['product_id'] == product_id, 'category']
    cat = cat_series.iloc[0] if len(cat_series) else None
    c_avg_qty = float(category_avg.get(cat, daily['daily_qty'].mean()))

    if p_hist.empty:
        # No sales history at all for this SKU -> category baseline is the
        # most honest estimate available (still real data, not a guess).
        forecast = [{"day": i, "forecast_qty": round(c_avg_qty, 1)} for i in range(1, forecast_days + 1)]
        return {
            "product_id": product_id, "forecast_horizon_days": forecast_days,
            "forecast": forecast, "historical": [],
            "mape": round(GLOBAL_MAPE, 2), "model": "Category Baseline (no sales history)",
        }

    last_date = p_hist['transaction_date'].iloc[-1]
    p_avg_qty = float(product_avg.get(product_id, daily['daily_qty'].mean()))
    recent_lags = list(p_hist['daily_qty'].tail(30))

    forecast = []
    for i in range(1, forecast_days + 1):
        future_date = last_date + pd.Timedelta(days=i)
        cal_info = holiday_lookup.get(future_date.normalize(), {})
        row = {
            'lag_7':  recent_lags[-7]  if len(recent_lags) >= 7  else recent_lags[-1],
            'lag_14': recent_lags[-14] if len(recent_lags) >= 14 else recent_lags[-1],
            'rolling_7_mean':  float(np.mean(recent_lags[-7:])),
            'rolling_30_mean': float(np.mean(recent_lags[-30:])),
            'dow_num': DOW_MAP.get(future_date.day_name(), 0),
            'month': future_date.month,
            'day_of_month': future_date.day,
            'week_number': int(future_date.isocalendar()[1]),
            'quarter': future_date.quarter,
            'is_holiday': int(cal_info.get('is_holiday', False)),
            'is_weekend': int(future_date.dayofweek >= 5),
            'is_sale_season': int(cal_info.get('is_sale_season', False)),
            'product_avg_qty': p_avg_qty,
            'category_avg_qty': c_avg_qty,
        }
        X_future = pd.DataFrame([row])[FEATURES]
        pred_qty = float(np.clip(model.predict(X_future)[0], 0, None))
        forecast.append({"day": i, "forecast_qty": round(pred_qty, 1)})
        recent_lags.append(pred_qty)

    historical = [
        {"date": str(r['transaction_date'].date()), "actual_qty": float(r['daily_qty'])}
        for _, r in p_hist.tail(14).iterrows()
    ]

    return {
        "product_id": product_id,
        "forecast_horizon_days": forecast_days,
        "forecast": forecast,
        "historical": historical,
        "mape": round(_product_test_mape(product_id), 2),
        "model": "XGBoost (Global Multi-Product Regressor)",
    }

sample_pid = daily['product_id'].iloc[0]
backend_res = get_demand_forecast(sample_pid, forecast_days=5)
print("Forecast payload:\n", json.dumps(backend_res, indent=2))

In [ ]:
import matplotlib.pyplot as plt

print("=== CALRETAIL DEMAND FORECAST TELEMETRY ===")
print(f"Product: {sample_pid} | Current Model Accuracy (100 - MAPE): {100 - backend_res['mape']:.1f}%")
print("Daily Forecast Units:")
for f in backend_res['forecast']:
    print(f"  Day {f['day']}: {f['forecast_qty']} units")

# Plot mock visualization
fig, ax = plt.subplots(figsize=(6, 3))
days = [x['day'] for x in backend_res['forecast']]
qtys = [x['forecast_qty'] for x in backend_res['forecast']]
ax.bar(days, qtys, color='skyblue')
ax.set_title(f"Forecast Horizon for {sample_pid}")
ax.set_xlabel("Day")
ax.set_ylabel("Predicted Quantities")
plt.show()
